In [ ]:
import sys
!{sys.executable} -m pip install nbformat>=4.2.0 ipywidgets scikit-learn kaleido

# EXP_009d1: Attractor Dominance — 125 Prompt Survey

## Depends On
**EXP_009d0 must PASS before running this notebook.**

## Hypotheses Under Test

### H1: Attractor Dominance
The `prolet` attractor is the dominant basin of GPT-2 Small's weight geometry.

### H2: Secondary Basin Existence
The `Divine` attractor is a genuine secondary basin, not a one-off artefact.

### H3: Dissolution Pathway Structure
The intermediate tokens reflect the statistical topology of the training corpus.

## Method
125 prompts across 7 categories (Complex, Narrative, Simple, Chemical, Acronyms, Vulgarity, Wild),
grouped by **independent variable** (syntactic register), NOT by predicted basin.
Early stopping when cosine similarity > 0.9999 for 3 consecutive iterations.
All outputs auto-saved to disk for automated review.

---


In [ ]:
# ============================================================
# STEP 1: SETUP
# ============================================================
import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
import os
import json
from datetime import datetime
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Running on: {device}")
print(f"Architecture: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")

# Output directories
os.makedirs('images', exist_ok=True)
os.makedirs('results', exist_ok=True)
print(f"Output dirs ready: images/, results/")

In [ ]:
# ============================================================
# STEP 2: LOAD PROMPT LIBRARY
# ============================================================
from prompt_library import (
    PROMPT_LIBRARY, PREDICTIONS, CATEGORY_MAP,
    COMPLEX, NARRATIVE, SIMPLE, CHEMICAL, ACRONYMS, VULGARITY, WILD
)

# Configuration
SNAPSHOT_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100]
MAX_ITERATIONS = 100
CONVERGENCE_THRESHOLD = 0.9999
CONVERGENCE_PATIENCE = 3  # consecutive iterations above threshold

LAYER_START = 0
LAYER_END = model.cfg.n_layers - 1

print(f"Loaded {len(PROMPT_LIBRARY)} prompts across {len(set(CATEGORY_MAP.values()))} categories")
print(f"Schedule: {SNAPSHOT_SCHEDULE}")
print(f"Early stopping: cosine_sim > {CONVERGENCE_THRESHOLD} for {CONVERGENCE_PATIENCE} consecutive iterations")
print(f"\nCategories:")
for cat in sorted(set(CATEGORY_MAP.values())):
    count = sum(1 for v in CATEGORY_MAP.values() if v == cat)
    print(f"  {cat}: {count} prompts")

In [ ]:
# ============================================================
# STEP 3: THE CORE ENGINE — With Early Stopping
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_total_resonance_loop(model, prompt, layer_start, layer_end, max_iter,
                             schedule, conv_threshold=0.9999, conv_patience=3):
    """Total Lucier Loop with norm normalisation, early stopping, and full-position decoding."""
    snapshots = []
    hook_point_read = f"blocks.{layer_end}.hook_resid_post"
    hook_point_write = f"blocks.{layer_start}.hook_resid_pre"
    converged_at = None
    consecutive_converged = 0
    
    with torch.no_grad():
        _, cache = model.run_with_cache(
            prompt,
            names_filter=lambda n: n == hook_point_read
        )
    
    current_tensor = cache[hook_point_read][0].clone()
    seq_len = current_tensor.shape[0]
    initial_norm = current_tensor.norm().item()
    
    last_vec = current_tensor[-1, :].clone()
    mean_vec = current_tensor.mean(dim=0).clone()
    
    if 0 in schedule:
        top_tokens_last = get_top_tokens(model, last_vec)
        all_pos_tokens = []
        for pos in range(seq_len):
            pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
            all_pos_tokens.append(pos_top[0][0])
        snapshots.append({
            "iteration": 0,
            "tensor": current_tensor.clone().cpu(),
            "last_vector": last_vec.clone().cpu(),
            "mean_vector": mean_vec.clone().cpu(),
            "last_norm": last_vec.norm().item(),
            "mean_norm": mean_vec.norm().item(),
            "tensor_norm": current_tensor.norm().item(),
            "top_tokens": top_tokens_last,
            "all_position_tokens": all_pos_tokens,
            "cosine_sim_last": 1.0,
            "cosine_sim_mean": 1.0,
            "position_similarity": 1.0,
        })
    
    prev_last = last_vec.clone()
    prev_mean = mean_vec.clone()
    
    for i in range(1, max_iter + 1):
        current_norm = current_tensor.norm().item()
        if current_norm > 0:
            current_tensor = current_tensor * (initial_norm / current_norm)
        
        inject_tensor = current_tensor.clone()
        
        def injection_hook(resid, hook, tensor=inject_tensor):
            resid[0, :, :] = tensor
            return resid
        
        model.add_hook(hook_point_write, injection_hook)
        try:
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    prompt,
                    names_filter=lambda n: n == hook_point_read
                )
        finally:
            model.reset_hooks()
        
        current_tensor = cache[hook_point_read][0].clone()
        last_vec = current_tensor[-1, :].clone()
        mean_vec = current_tensor.mean(dim=0).clone()
        
        # Check convergence
        cos_sim_mean = torch.nn.functional.cosine_similarity(
            mean_vec.unsqueeze(0), prev_mean.unsqueeze(0)
        ).item()
        
        if cos_sim_mean > conv_threshold:
            consecutive_converged += 1
        else:
            consecutive_converged = 0
        
        should_snapshot = (i in schedule) or (consecutive_converged == conv_patience and converged_at is None)
        
        if should_snapshot:
            cos_sim_last = torch.nn.functional.cosine_similarity(
                last_vec.unsqueeze(0), prev_last.unsqueeze(0)
            ).item()
            
            pos_norms = current_tensor.norm(dim=1, keepdim=True).clamp(min=1e-8)
            normalized_positions = current_tensor / pos_norms
            pos_sim_matrix = normalized_positions @ normalized_positions.T
            mask = ~torch.eye(seq_len, dtype=torch.bool, device=pos_sim_matrix.device)
            position_similarity = pos_sim_matrix[mask].mean().item()
            
            top_tokens_last = get_top_tokens(model, last_vec)
            all_pos_tokens = []
            for pos in range(seq_len):
                pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
                all_pos_tokens.append(pos_top[0][0])
            
            snapshots.append({
                "iteration": i,
                "tensor": current_tensor.clone().cpu(),
                "last_vector": last_vec.clone().cpu(),
                "mean_vector": mean_vec.clone().cpu(),
                "last_norm": last_vec.norm().item(),
                "mean_norm": mean_vec.norm().item(),
                "tensor_norm": current_tensor.norm().item(),
                "top_tokens": top_tokens_last,
                "all_position_tokens": all_pos_tokens,
                "cosine_sim_last": cos_sim_last,
                "cosine_sim_mean": cos_sim_mean,
                "position_similarity": position_similarity,
            })
            print(f"  iter {i:>3}: top='{top_tokens_last[0][0].strip()}', "
                  f"cos_mean={cos_sim_mean:.6f}, pos_collapse={position_similarity:.4f}")
        
        # Early stopping
        if consecutive_converged >= conv_patience and converged_at is None:
            converged_at = i
            print(f"  ✓ CONVERGED at iteration {i} (cos_sim > {conv_threshold} for {conv_patience} consecutive)")
            break
        
        prev_last = last_vec.clone()
        prev_mean = mean_vec.clone()
    
    if converged_at is None:
        print(f"  ⚠ Did NOT converge within {max_iter} iterations")
    
    return snapshots, converged_at

print("Engine loaded (with early stopping).")

In [ ]:
# ============================================================
# STEP 4: RUN ALL 125 PROMPTS
# ============================================================
import time

all_results = {}
convergence_points = {}
start_time = time.time()

for idx, (label, prompt) in enumerate(PROMPT_LIBRARY.items()):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(PROMPT_LIBRARY)}] '{label}' — \"{prompt[:50]}...\"")
    print(f"{'='*60}")
    
    snapshots, conv_at = run_total_resonance_loop(
        model, prompt,
        layer_start=LAYER_START,
        layer_end=LAYER_END,
        max_iter=MAX_ITERATIONS,
        schedule=SNAPSHOT_SCHEDULE,
        conv_threshold=CONVERGENCE_THRESHOLD,
        conv_patience=CONVERGENCE_PATIENCE
    )
    all_results[label] = snapshots
    convergence_points[label] = conv_at

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"ALL {len(PROMPT_LIBRARY)} RECORDINGS COMPLETE in {elapsed/60:.1f} minutes.")
converged_count = sum(1 for v in convergence_points.values() if v is not None)
print(f"Converged: {converged_count}/{len(PROMPT_LIBRARY)}")
avg_conv = np.mean([v for v in convergence_points.values() if v is not None])
print(f"Average convergence iteration: {avg_conv:.1f}")

In [ ]:
# ============================================================
# VIS 5a: PREDICTION RESULTS — By Category
# ============================================================

md = "## Prediction Results — 125 Prompt Survey\n\n"
md += f"**Run date:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n"

# Summary by category
md += "### Summary by Category\n\n"
md += "| Category | N | Basins Found | Most Common | Convergence (avg iter) |\n"
md += "|:---|:---|:---|:---|:---|\n"

for cat in ['Complex', 'Narrative', 'Simple', 'Chemical', 'Acronyms', 'Vulgarity', 'Wild']:
    cat_labels = [k for k, v in CATEGORY_MAP.items() if v == cat]
    terminals = []
    conv_iters = []
    for label in cat_labels:
        if label in all_results and len(all_results[label]) > 0:
            tok = all_results[label][-1]['top_tokens'][0][0].strip()
            terminals.append(tok)
        if convergence_points.get(label) is not None:
            conv_iters.append(convergence_points[label])
    
    from collections import Counter
    counts = Counter(terminals)
    basins_str = ', '.join(f'`{t}`({c})' for t, c in counts.most_common(5))
    most_common = counts.most_common(1)[0][0] if counts else '—'
    avg_c = f"{np.mean(conv_iters):.0f}" if conv_iters else '—'
    md += f"| {cat} | {len(cat_labels)} | {basins_str} | `{most_common}` | {avg_c} |\n"

md += "\n"

# Full table
md += "### Full Results\n\n"
md += "| ID | Category | Predicted | Actual Terminal | Conv. Iter | Match? |\n"
md += "|:---|:---|:---|:---|:---|:---|\n"

basin_counts = Counter()

for label in PROMPT_LIBRARY.keys():
    if label not in all_results or len(all_results[label]) == 0:
        continue
    terminal = all_results[label][-1]['top_tokens'][0][0].strip()
    predicted, confidence = PREDICTIONS.get(label, ('unknown', 'none'))
    cat = CATEGORY_MAP.get(label, '?')
    conv = convergence_points.get(label, '—')
    
    basin_counts[terminal] += 1
    
    if predicted == 'unknown':
        match = '—'
    elif predicted.lower() in terminal.lower() or terminal.lower() in predicted.lower():
        match = '✓'
    else:
        match = '✗'
    
    md += f"| {label} | {cat} | `{predicted}` ({confidence}) | `{terminal}` | {conv} | {match} |\n"

md += f"\n### Basin Distribution\n\n"
for tok, count in basin_counts.most_common():
    pct = count / len(all_results) * 100
    md += f"- `{tok}`: {count} prompts ({pct:.1f}%)\n"

display(Markdown(md))

# Auto-save
with open('results/d1_predictions.md', 'w', encoding='utf-8') as f:
    f.write(md)
print("[SAVED] results/d1_predictions.md")

In [ ]:
# ============================================================
# VIS 5b: CROSS-PROMPT CONVERGENCE MATRIX
# ============================================================

labels = list(all_results.keys())
n = len(labels)
sim_matrix = np.zeros((n, n))

final_vectors = []
for label in labels:
    final_vec = all_results[label][-1]["mean_vector"]
    final_vectors.append(final_vec)

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = torch.nn.functional.cosine_similarity(
            final_vectors[i].unsqueeze(0).float(),
            final_vectors[j].unsqueeze(0).float()
        ).item()

# Color-code by category
cat_labels = [f"{CATEGORY_MAP.get(l, '?')[0]}:{l}" for l in labels]

fig_sim = px.imshow(
    sim_matrix,
    x=cat_labels, y=cat_labels,
    color_continuous_scale="Viridis",
    title="EXP_009d1: Cross-Prompt Convergence Matrix (125 Prompts)",
    text_auto=".2f",
    aspect="auto",
)
fig_sim.update_layout(template="plotly_dark", height=1200, width=1200,
                      font=dict(size=6))
fig_sim.show()
fig_sim.write_image('images/d1_convergence_matrix.png', scale=2)
print("[SAVED] images/d1_convergence_matrix.png")

In [ ]:
# ============================================================
# VIS 5c: 3D PCA TRAJECTORIES — All 125 Prompts
# ============================================================
from sklearn.decomposition import PCA
import pandas as pd

all_vecs = []
labels_list = []
iters_list = []
cats_list = []
text_list = []

for label, snapshots in all_results.items():
    for s in snapshots:
        all_vecs.append(s["mean_vector"].detach().cpu().numpy())
        labels_list.append(label)
        iters_list.append(s["iteration"])
        cats_list.append(CATEGORY_MAP.get(label, '?'))
        top_tok = s['top_tokens'][0][0].replace('\n', '↵').strip()
        text_list.append(f"{label} iter {s['iteration']}: {top_tok}")

all_vecs = np.array(all_vecs)
pca = PCA(n_components=3)
vecs_3d = pca.fit_transform(all_vecs)

df = pd.DataFrame({
    'x': vecs_3d[:, 0],
    'y': vecs_3d[:, 1],
    'z': vecs_3d[:, 2],
    'Prompt': labels_list,
    'Category': cats_list,
    'Iteration': iters_list,
    'Top_Token': text_list
})

fig_topo = px.line_3d(
    df, x='x', y='y', z='z',
    color='Category',
    hover_name='Top_Token',
    markers=True,
    title=f"EXP_009d1: Attractor Landscape — 125 Prompt Trajectories<br>"
          f"<sup>(Explained Variance: {sum(pca.explained_variance_ratio_)*100:.1f}%)</sup>"
)
fig_topo.update_traces(marker=dict(size=3), line=dict(width=2))
fig_topo.update_layout(
    template="plotly_dark",
    height=900,
    scene=dict(
        xaxis_title="PC 1",
        yaxis_title="PC 2",
        zaxis_title="PC 3",
    )
)
fig_topo.show()
fig_topo.write_image('images/d1_topology_125.png', scale=2)
print("[SAVED] images/d1_topology_125.png")

In [ ]:
# ============================================================
# VIS 5d: BASIN PIE CHART — What fraction goes where?
# ============================================================
from collections import Counter

terminals = []
for label, snapshots in all_results.items():
    tok = snapshots[-1]['top_tokens'][0][0].strip()
    terminals.append(tok)

basin_counts = Counter(terminals)

fig_pie = px.pie(
    names=list(basin_counts.keys()),
    values=list(basin_counts.values()),
    title="EXP_009d1: Basin Distribution — 125 Prompts",
    hole=0.3
)
fig_pie.update_layout(template="plotly_dark", height=500)
fig_pie.show()
fig_pie.write_image('images/d1_basin_distribution.png', scale=2)
print("[SAVED] images/d1_basin_distribution.png")

In [ ]:
# ============================================================
# VIS 5e: CONVERGENCE SPEED BY CATEGORY
# ============================================================

conv_data = []
for label, conv_at in convergence_points.items():
    if conv_at is not None:
        conv_data.append({
            'Prompt': label,
            'Category': CATEGORY_MAP.get(label, '?'),
            'Converged_At': conv_at
        })

df_conv = pd.DataFrame(conv_data)

fig_conv = px.box(
    df_conv, x='Category', y='Converged_At',
    color='Category',
    title='EXP_009d1: Convergence Speed by Category',
    labels={'Converged_At': 'Iteration at Convergence'},
    points='all'
)
fig_conv.update_layout(template='plotly_dark', height=500)
fig_conv.show()
fig_conv.write_image('images/d1_convergence_speed.png', scale=2)
print("[SAVED] images/d1_convergence_speed.png")

In [ ]:
# ============================================================
# STEP 6: SAVE ALL ARTIFACTS
# ============================================================

save_dir = os.path.join("..", "_DATA", "EXP_009")
os.makedirs(save_dir, exist_ok=True)

save_data = {}
for label, snapshots in all_results.items():
    save_data[label] = {
        "iterations": [s["iteration"] for s in snapshots],
        "last_vectors": torch.stack([s["last_vector"] for s in snapshots]),
        "mean_vectors": torch.stack([s["mean_vector"] for s in snapshots]),
        "last_norms": [s["last_norm"] for s in snapshots],
        "mean_norms": [s["mean_norm"] for s in snapshots],
        "cosine_sims_last": [s["cosine_sim_last"] for s in snapshots],
        "cosine_sims_mean": [s["cosine_sim_mean"] for s in snapshots],
        "position_similarity": [s["position_similarity"] for s in snapshots],
        "top_tokens": [s["top_tokens"] for s in snapshots],
        "all_position_tokens": [s["all_position_tokens"] for s in snapshots],
        "converged_at": convergence_points.get(label),
    }

torch.save(save_data, os.path.join(save_dir, "009d1_125prompt_results.pt"))
print(f"[SAVED] {save_dir}/009d1_125prompt_results.pt")

config = {
    "schedule": SNAPSHOT_SCHEDULE,
    "max_iterations": MAX_ITERATIONS,
    "convergence_threshold": CONVERGENCE_THRESHOLD,
    "convergence_patience": CONVERGENCE_PATIENCE,
    "layer_start": LAYER_START,
    "layer_end": LAYER_END,
    "model": "gpt2-small",
    "n_prompts": len(PROMPT_LIBRARY),
    "categories": list(set(CATEGORY_MAP.values())),
}
torch.save(config, os.path.join(save_dir, "009d1_125prompt_config.pt"))
print(f"[SAVED] {save_dir}/009d1_125prompt_config.pt")

# Save summary JSON (readable without torch)
summary = {
    "run_date": datetime.now().isoformat(),
    "n_prompts": len(PROMPT_LIBRARY),
    "converged": sum(1 for v in convergence_points.values() if v is not None),
    "results": {}
}
for label in PROMPT_LIBRARY:
    if label in all_results and len(all_results[label]) > 0:
        summary["results"][label] = {
            "category": CATEGORY_MAP.get(label),
            "terminal_token": all_results[label][-1]['top_tokens'][0][0].strip(),
            "converged_at": convergence_points.get(label),
            "predicted": PREDICTIONS.get(label, ['unknown'])[0],
        }

with open('results/d1_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("[SAVED] results/d1_summary.json")

print(f"\n✓ All artifacts saved. Ready for review.")